# Exercise 1: PydanticAI, FastAPI

## 0. PydanticAI fundamentals

Make a PydanticAI model that can take an input of a location and then it should suggest 5 restaurants nearby that place. The restaurant model should have

- name
- type of food (cuisine)
- price level
- rating
- short description
- opening hours
- location

It's okay if your model is making up a restaurant that doesn't exist

In [ ]:
# model setup
#
# Sets up the connection to the LLM used by every agent in this notebook.
# NVIDIA's API speaks the same "chat completions" format as OpenAI, so we
# reuse the OpenAI classes and just point them at NVIDIA's address instead
# (same pattern as exercise_0).

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

import os
from dotenv import load_dotenv

# Depends on the .env file in the project root: load_dotenv() reads its
# key=value pairs (e.g. NVIDIA_API=...) into environment variables, so the
# key never has to be typed or hard-coded into this notebook.
load_dotenv()

model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # which model to call
    provider = OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1",  # NVIDIA's endpoint, not OpenAI's
        api_key = os.environ["NVIDIA_API"],  # read from .env, never hard-coded
    ),

)

In [ ]:
# restaurant model
#
# The "shape" we want the LLM's answer to have. Used below as the agent's
# output_type (when set), so PydanticAI checks every field is present and
# of the right type before we ever see the result - e.g. a rating outside
# 0-5 gets rejected and sent back to the model for a retry.
from pydantic import BaseModel, Field

class Restaurant(BaseModel):
    """A single restaurant suggestion"""

    name: str = Field(description= "Restaurant's name")
    cuisine: str = Field(description= "Type of food served, e.g. Italian, Sushi, Vegan")
    price_level: str = Field(description = "Price level as $, $$, $$$ or $$$$")
    # ge/le = "greater-or-equal" / "less-or-equal": restricts valid ratings to 0-5.
    rating: float = Field(ge=0, le=5, description="Rating out of 5")
    description: str = Field(description="A short one or two sentence description")
    opening_hours: str = Field(description="Opening hours, e.g. 'Mon-Sat 11:00-22:00'")
    location: str = Field(description="Street address or neighborhood of the restaurant")


In [ ]:
# Agent using list [Restaurant] as output type
#
# An Agent wraps the model (from the setup cell above) with `instructions`,
# the system prompt that applies on every call. Depends on `model` (model
# setup cell) and `Restaurant` (model cell above this one).
#
# NOTE: no output_type is set here, so result.output below will be plain
# text (a str) that just looks like it lists restaurants - not a validated
# list[Restaurant], even though the function's type hint claims that.
restaurant_agent = Agent(
    model,
    instructions = (
        "You suggest restaurants near a given location. "
        "Given a location, suggest exactly 5 restaurants near it. "
        "It is fine to invent plausible restaurants that may not actually exist. "
        "Fill in every field for each restaurant."
    ),
)

# function to suggest restaurant
async def suggest_restaurants(location: str) -> list[Restaurant]:
    """Suggest 5 restaurants near a location.

    Args:
        location: A place name, e.g. "Gamla Stan, Stockholm".

    Returns:
        A list of 5 validated Restaurant objects.
    """
    # await: agent.run() is async because it waits on the network call to the
    # LLM. model_settings={"timeout": 120} raises how long we wait before giving up.
    result = await restaurant_agent.run(location, model_settings = {"timeout":120})
    # Whatever the model answered with, unvalidated and untouched.
    return result.output

In [ ]:
# Call
#
# Depends on suggest_restaurants() and restaurant_agent (previous cell) and
# Restaurant (model cell). Flow: location string -> suggest_restaurants()
# -> agent.run() -> result.output -> here, printed per restaurant.
restaurants = await suggest_restaurants("Gamla Stan, Stockholm")

for r in restaurants:
    # Redefines restaurant_agent here, now WITH output_type=list[Restaurant] -
    # this re-runs on every iteration of this outer loop, which is wasteful.
    restaurant_agent = Agent(
        model,
        output_type=list[Restaurant],
        instructions=(
            "You suggest restaurants near a given location. "
            "Given a location, suggest exactly 5 restaurants near it. "
            "It is fine to invent plausible restaurants that may not actually exist. "
            "Fill in every field for each restaurant."
        ),
    )

    restaurants = await suggest_restaurants("Gamla Stan, Stockholm")

    # Even with output_type=list[Restaurant] set, `restaurants` here can still
    # come back as plain text rather than Restaurant objects - many providers
    # need the structured-output schema to be a JSON object, not a raw list.
    # When that happens, iterating a str yields individual characters, so
    # `r` is a single character below, not a Restaurant - hence r.name fails.
    for r in restaurants:
        print(f" {r.name} | {r.cuisine} | {r.price_level} | {r.rating}/5")
        print(f"  {r.description}")
        print(f"  Hours: {r.opening_hours} | Location: {r.location}")
    print(f"  {r.description}")
    print(f"  Hours: {r.opening_hours} | Location: {r.location}")